# Sequential GBT to LLM rerank from scratch

This notebook demonstrates the sequential pipeline: first use a GBT-style scorer to retrieve a short list, then rerank only that short list with an LLM-style contextual scorer.

In [1]:
from math import log2

user = {"likes": {"coffee", "brunch", "vietnamese"}, "needs": {"quiet", "nearby"}, "price": 2, "max_distance": 4.0}
candidates = [
    {"id": "b1", "name": "Morning Bean", "tags": {"coffee", "bakery", "quiet"}, "price": 2, "distance": 0.7, "rating": 4.7, "reviews": 410, "label": 3},
    {"id": "b2", "name": "Pho Corner", "tags": {"vietnamese", "noodles", "casual"}, "price": 1, "distance": 2.2, "rating": 4.5, "reviews": 260, "label": 3},
    {"id": "b3", "name": "Quiet Study Cafe", "tags": {"coffee", "dessert", "quiet"}, "price": 2, "distance": 1.1, "rating": 4.1, "reviews": 95, "label": 2},
    {"id": "b4", "name": "Late Night BBQ", "tags": {"bbq", "korean", "loud"}, "price": 3, "distance": 6.5, "rating": 4.6, "reviews": 520, "label": 0},
    {"id": "b5", "name": "Campus Banh Mi", "tags": {"vietnamese", "sandwich", "cheap"}, "price": 1, "distance": 0.4, "rating": 4.2, "reviews": 140, "label": 2},
    {"id": "b6", "name": "Premium Steak Loft", "tags": {"steak", "wine", "expensive"}, "price": 4, "distance": 3.6, "rating": 4.8, "reviews": 310, "label": 1},
    {"id": "b7", "name": "Weekend Brunch Lab", "tags": {"brunch", "coffee", "popular"}, "price": 2, "distance": 3.2, "rating": 4.4, "reviews": 180, "label": 3},
    {"id": "b8", "name": "Burger Dock", "tags": {"burger", "fast food"}, "price": 2, "distance": 1.8, "rating": 3.9, "reviews": 220, "label": 1},
]
print(candidates)

[{'id': 'b1', 'name': 'Morning Bean', 'tags': {'coffee', 'quiet', 'bakery'}, 'price': 2, 'distance': 0.7, 'rating': 4.7, 'reviews': 410, 'label': 3}, {'id': 'b2', 'name': 'Pho Corner', 'tags': {'vietnamese', 'noodles', 'casual'}, 'price': 1, 'distance': 2.2, 'rating': 4.5, 'reviews': 260, 'label': 3}, {'id': 'b3', 'name': 'Quiet Study Cafe', 'tags': {'coffee', 'quiet', 'dessert'}, 'price': 2, 'distance': 1.1, 'rating': 4.1, 'reviews': 95, 'label': 2}, {'id': 'b4', 'name': 'Late Night BBQ', 'tags': {'bbq', 'korean', 'loud'}, 'price': 3, 'distance': 6.5, 'rating': 4.6, 'reviews': 520, 'label': 0}, {'id': 'b5', 'name': 'Campus Banh Mi', 'tags': {'sandwich', 'vietnamese', 'cheap'}, 'price': 1, 'distance': 0.4, 'rating': 4.2, 'reviews': 140, 'label': 2}, {'id': 'b6', 'name': 'Premium Steak Loft', 'tags': {'expensive', 'wine', 'steak'}, 'price': 4, 'distance': 3.6, 'rating': 4.8, 'reviews': 310, 'label': 1}, {'id': 'b7', 'name': 'Weekend Brunch Lab', 'tags': {'coffee', 'brunch', 'popular'}, 

In [2]:
def gbt_score(user, item):
    category = len(user["likes"] & item["tags"]) / len(user["likes"])
    price = 1 - min(abs(user["price"] - item["price"]), 3) / 3
    distance = max(0, 1 - item["distance"] / user["max_distance"])
    popularity = min(item["reviews"], 500) / 500
    return 0.42 * category + 0.18 * price + 0.18 * distance + 0.14 * (item["rating"] / 5) + 0.08 * popularity

first_pass = sorted(
    [{**item, "gbt_score": round(gbt_score(user, item), 4)} for item in candidates],
    key=lambda row: row["gbt_score"],
    reverse=True,
)
shortlist = first_pass[:5]
[(row["id"], row["name"], row["gbt_score"], row["label"]) for row in shortlist]


[('b1', 'Morning Bean', 0.6657, 3),
 ('b7', 'Weekend Brunch Lab', 0.648, 3),
 ('b3', 'Quiet Study Cafe', 0.5805, 2),
 ('b5', 'Campus Banh Mi', 0.562, 2),
 ('b2', 'Pho Corner', 0.5086, 3)]

In [3]:
def llm_rerank_score(user, item):
    score = 0.0
    score += 0.35 if user["likes"] & item["tags"] else 0.0
    score += 0.25 if user["needs"] & item["tags"] else 0.0
    score += 0.25 if item["distance"] <= 1.5 else 0.05
    score += 0.10 if item["price"] <= 2 else -0.15
    score -= 0.30 if "loud" in item["tags"] or "expensive" in item["tags"] else 0.0
    return score

reranked_shortlist = sorted(
    [{**item, "llm_score": round(llm_rerank_score(user, item), 4)} for item in shortlist],
    key=lambda row: row["llm_score"],
    reverse=True,
)

shortlist_ids = {item["id"] for item in reranked_shortlist}
tail = [item for item in first_pass if item["id"] not in shortlist_ids]
final_ranked = reranked_shortlist + tail
[(row["id"], row["name"], row.get("llm_score"), row["gbt_score"], row["label"]) for row in final_ranked]


[('b1', 'Morning Bean', 0.95, 0.6657, 3),
 ('b3', 'Quiet Study Cafe', 0.95, 0.5805, 2),
 ('b5', 'Campus Banh Mi', 0.7, 0.562, 2),
 ('b7', 'Weekend Brunch Lab', 0.5, 0.648, 3),
 ('b2', 'Pho Corner', 0.5, 0.5086, 3),
 ('b8', 'Burger Dock', None, 0.4234, 1),
 ('b4', 'Late Night BBQ', None, 0.3288, 0),
 ('b6', 'Premium Steak Loft', None, 0.262, 1)]

In [4]:
def dcg(labels):
    return sum((2 ** label - 1) / log2(rank + 2) for rank, label in enumerate(labels))

def ndcg(rows, k):
    actual = [row["label"] for row in rows[:k]]
    ideal = sorted((row["label"] for row in rows), reverse=True)[:k]
    return dcg(actual) / dcg(ideal) if dcg(ideal) else 0.0

for name, rows in [("first pass", first_pass), ("sequential", final_ranked)]:
    print(name)
    print("ndcg@3", round(ndcg(rows, 3), 4), "ndcg@5", round(ndcg(rows, 5), 4), "ndcg@10", round(ndcg(rows, 10), 4))


first pass
ndcg@3 0.8659 ndcg@5 0.9739 ndcg@10 0.9739
sequential
ndcg@3 0.6967 ndcg@5 0.9278 ndcg@10 0.9296
